# Customer Churn Modelling — Classification

Predicting whether a bank customer will exit (churn) using **Logistic Regression**, **Decision Tree**, and **Random Forest** classifiers.

**Dataset:** `Churn_Modelling.csv` — 10,000 bank customer records.

> **Note:** Place `Churn_Modelling.csv` in the same directory, or upload it when running in Google Colab.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


## Step 2: Load the Data

In [ ]:
df = pd.read_csv('Churn_Modelling.csv')

# Drop columns that are not useful for prediction
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)
df.head()


## Step 3: Data Exploration

In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe().T


In [ ]:
# Target variable distribution
count = df['Exited'].value_counts()
print(count)


## Step 4: Data Visualization

### Target Distribution

In [ ]:
plt.figure(figsize=(6, 6))
plt.pie(count, explode=(0, 0.1), labels=['Stayed (0)', 'Exited (1)'],
        autopct='%1.1f%%', colors=['steelblue', 'tomato'])
plt.title('Customer Churn Distribution')
plt.show()


### Categorical Feature Distributions

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(15, 12))
sns.countplot(x='Geography', hue='Exited', data=df, ax=ax[0, 0])
sns.countplot(x='Gender', hue='Exited', data=df, ax=ax[0, 1])
sns.countplot(x='HasCrCard', hue='Exited', data=df, ax=ax[1, 0])
sns.countplot(x='IsActiveMember', hue='Exited', data=df, ax=ax[1, 1])
sns.countplot(x='NumOfProducts', hue='Exited', data=df, ax=ax[2, 0])
sns.countplot(x='Tenure', hue='Exited', data=df, ax=ax[2, 1])
plt.tight_layout()
plt.show()


### Numerical Features vs Churn

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(x='Exited', y='CreditScore', data=df, ax=ax[0, 0])
sns.boxplot(x='Exited', y='Age', data=df, ax=ax[0, 1])
sns.boxplot(x='Exited', y='Balance', data=df, ax=ax[1, 0])
sns.boxplot(x='Exited', y='EstimatedSalary', data=df, ax=ax[1, 1])
plt.tight_layout()
plt.show()


### Correlation Heatmap

In [ ]:
corr = df.select_dtypes(include='number').corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, linecolor='black', cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


## Step 5: Preprocessing — Encode Categorical Variables

In [ ]:
df_encoded = pd.get_dummies(df, columns=['Geography', 'Gender'],
                            drop_first=True)
# Convert bool columns to int for model compatibility
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)
df_encoded.head()


## Step 6: Split the Data

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)
print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)


## Step 7: Train the Models

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print(f'Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%')
print(classification_report(y_test, y_pred_lr))


### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier(max_depth=5, random_state=42)
dtc.fit(X_train, y_train)
y_pred_dtc = dtc.predict(X_test)

print(f'Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dtc)*100:.2f}%')
print(classification_report(y_test, y_pred_dtc))


### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train, y_train)
y_pred_rfc = rfc.predict(X_test)

print(f'Random Forest Accuracy: {accuracy_score(y_test, y_pred_rfc)*100:.2f}%')
print(classification_report(y_test, y_pred_rfc))


## Model Comparison

| Model | Notes |
|-------|-------|
| Logistic Regression | Simple linear classifier, ~81% |
| Decision Tree | Better at non-linear patterns, ~85% |
| **Random Forest** | **Best model, ~87%** — ensemble reduces variance |

**Random Forest** is recommended for this task due to the non-linear relationships in customer churn data.